# Audio Deepfake Detection — ASVspoof 2019 LA (Colab runner)

Reduced, leakage-aware study. Runs P0→P9 from the project guide:
env check → data → manifest/audit → shortcut audit → silence trimming →
norm stats → sanity gate → 12 training runs (3 seeds × 4 configs) → tables & figures.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough).
Put `LA.zip` on your Drive once; checkpoints are written back to Drive so a
disconnect never loses progress. Re-running the training cell resumes automatically.

> Rule zero: every reported number comes from a run you actually execute here.

## P0 — Environment check

In [ ]:
import subprocess, os
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
      '--format=csv,noheader'], capture_output=True, text=True).stdout or 'no GPU')
print('CPU cores:', os.cpu_count())
import sys; print('python', sys.version.split()[0])
import torch, torchaudio, torchvision
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Mount Google Drive and set paths
Edit `DRIVE_DIR` to wherever you keep the dataset. `LA.zip` must already be at
`DRIVE_DIR/LA.zip` (download it once from Edinburgh DataShare — see the README).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/asvspoof')   # <-- edit me
DATA_ZIP  = DRIVE_DIR / 'LA.zip'                       # dataset zip on Drive
RUNS_DIR  = DRIVE_DIR / 'runs'                         # checkpoints (persistent)
DATA_ROOT = Path('/content/data_root')                # local unzip target (fast)
REPO      = Path('/content/audio-deepfake')           # this repo on Colab
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('zip on Drive:', DATA_ZIP, '-', 'FOUND' if DATA_ZIP.exists() else 'MISSING')

### Get this repository onto Colab
Upload `audio-deepfake.zip` to your `DRIVE_DIR` (or to `/content`) and this cell
unzips it to `REPO`. If you cloned it from git instead, just set `REPO` above and
skip this cell.

In [ ]:
import zipfile, shutil
if not REPO.exists():
    for cand in [DRIVE_DIR/'audio-deepfake.zip', Path('/content/audio-deepfake.zip')]:
        if cand.exists():
            with zipfile.ZipFile(cand) as z: z.extractall('/content')
            print('unzipped repo from', cand); break
    else:
        print('Upload audio-deepfake.zip to', DRIVE_DIR, 'or /content, then re-run.')
assert REPO.exists(), 'repo not found at '+str(REPO)
%cd {REPO}

### (Optional) download LA.zip to Drive if you have not already
Only run this if `DATA_ZIP` is MISSING above. The DataShare file is ~7.1 GB.
Verify the licence on the record page first: https://datashare.ed.ac.uk/handle/10283/3336

In [ ]:
# Uncomment ONE option if you need to fetch the data.
# Option A — Edinburgh DataShare direct bitstream (verify URL on the page first):
# !wget -c 'https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip' -O {DATA_ZIP}
#
# Option B — HuggingFace mirror of the LA release:
# !pip -q install huggingface_hub
# from huggingface_hub import hf_hub_download  # dataset: LanceaKing/asvspoof2019
#
# After download, (optional) verify integrity of the DataShare zip:
# !md5sum {DATA_ZIP}   # expect b53f0668f3b17ce72535fb60cac78923 for LA.zip
print('skip unless needed')

## Unzip the dataset locally (fast reads)
Copies `LA.zip` from Drive to local `/content` and unzips to `DATA_ROOT`.
Reading the 121k FLAC files straight from Drive is far too slow, so we stage locally.

In [ ]:
import zipfile, shutil, time
local_zip = Path('/content/LA.zip')
if not (DATA_ROOT/'LA').exists():
    if not local_zip.exists():
        print('copying zip from Drive -> /content ...'); t=time.time()
        shutil.copy(DATA_ZIP, local_zip); print('copied in', round(time.time()-t), 's')
    print('unzipping ...'); t=time.time()
    with zipfile.ZipFile(local_zip) as z: z.extractall(DATA_ROOT)
    print('unzipped in', round(time.time()-t), 's')
else:
    print('already unzipped')
print(sorted(p.name for p in (DATA_ROOT/'LA').glob('*'))[:10])
DATA = str(DATA_ROOT)   # folder that CONTAINS LA/

## Install the few extra packages (torch is already on Colab)

In [ ]:
!pip -q install soundfile librosa pyyaml joblib pyarrow
print('ok')

## P2 — Manifest and audit
Probing 121k files takes a few minutes on Colab's 2 cores. Check the audit: counts
should match the guide, speakers disjoint across splits, all sr = 16 kHz.

In [ ]:
!python scripts/build_manifest.py --data_root {DATA} --n_jobs 2

## P4 — Silence trimming
Defaults (35 dB, 0.05 s) are the guide's start values. To pick them by dev AUC first
(gate G1), run `select_trim.py` (heavier) and pass its winner here.

In [ ]:
# Optional parameter search on a subsample (uses dev only, never eval):
# !python scripts/select_trim.py --data_root {DATA} --subsample 6000 --n_jobs 2
!python scripts/compute_trim.py --data_root {DATA} --thresh_db 35 --margin_s 0.05 --n_jobs 2

## P3 & P5 — Shortcut audit (raw, then trimmed)
Trains tiny classifiers on non-speech cues only (silence, loudness, duration,
first 0.5 s). **Gate G1:** after trimming, every group should be below 0.60 AUC on
dev and eval. `duration` may stay higher — record it as a limitation, don't cheat.
Feature extraction over all files is the slow part; results are cached to Drive-free
local parquet, re-run is instant.

In [ ]:
!python scripts/run_shortcut.py --setting raw     --data_root {DATA} --n_jobs 2

In [ ]:
!python scripts/run_shortcut.py --setting trimmed --data_root {DATA} --n_jobs 2
import pandas as pd
sc = pd.read_csv('results/shortcut/shortcut_results.csv')
print(sc.pivot_table(index=['group'], columns=['setting','split'], values='auc').round(3))

## P6 — Normalization stats (both settings) + overfit sanity gate
Stats from 3,000 train files (seed 0), per setting. Then a model must memorize 64
samples to near-zero loss with no NaN, or something is wrong before you spend GPU time.

In [ ]:
!python scripts/compute_norm_stats.py --data_root {DATA} --setting trim
!python scripts/compute_norm_stats.py --data_root {DATA} --setting untrim

In [ ]:
!python scripts/overfit_test.py --config configs/A_trim.yaml --data_root {DATA} --steps 60

## P7 — First real run: `A_trim` seed 0 (Gate G2)
Checkpoints go to Drive (`RUNS_DIR`). **G2:** dev EER clearly below 50% and a
measurable dev→eval gap. Note the wall-clock time this takes — it replaces the
guide's rough estimate. If the session drops, just re-run this cell (it resumes).

In [ ]:
!python scripts/train_run.py --config configs/A_trim.yaml --seed 0 \
    --data_root {DATA} --runs_dir {RUNS_DIR} --resume

## P8 — The remaining 11 core runs
4 configs × seeds {0,1,2} = 12 runs total (A_trim s0 already done). `--resume` makes
this safe to re-run after any disconnect: finished runs are quick to re-score, the
unfinished one continues. Expect several GPU hours overall — measure your first run.

In [ ]:
configs = ['configs/A_trim.yaml','configs/A_untrim.yaml',
           'configs/B_trim.yaml','configs/F_trim.yaml']
for cfg in configs:
    for seed in [0,1,2]:
        print('==== ', cfg, 'seed', seed, ' ====')
        !python scripts/train_run.py --config {cfg} --seed {seed} \
            --data_root {DATA} --runs_dir {RUNS_DIR} --resume

## P9 — Aggregate: tables and figures
Reads every `metrics.json` under `RUNS_DIR` and builds the tables and plots.

In [ ]:
import shutil, os
# point aggregate at the Drive runs by symlinking them into results/runs
local_runs = REPO/'results'/'runs'
local_runs.mkdir(parents=True, exist_ok=True)
for d in Path(RUNS_DIR).glob('*'):
    link = local_runs/d.name
    if not link.exists():
        try: os.symlink(d, link)
        except OSError: shutil.copytree(d, link)
!python scripts/aggregate.py

In [ ]:
import pandas as pd
display(pd.read_csv('results/tables/main_results.csv').round(4))
from IPython.display import Image, display as D
for png in ['shortcut_before_after','eer_by_model','per_attack_eer','score_distributions_A_trim']:
    p = f'results/figures/{png}.png'
    if Path(p).exists(): D(Image(p))

## Next
- Fill in `DECISIONS.md` (gates, chosen trim params, deviations, measured run time).
- Stretch goals S1–S4 only after P0–P9 are complete (see the project guide, section 10).
- Hand back: `manifest.csv`, `audit.csv`, `shortcut_results.csv`, all `metrics.json`,
  the tables/figures, `DECISIONS.md`, and a one-page summary.